# Dataset: CoNLL-2003
* **Introduction to the CoNLL-2003 Shared Task:Language-Independent Named Entity Recognition:** https://www.aclweb.org/anthology/W03-0419.pdf
* **Named Entity Recognition (NER) with BERT in Spark NLP (cf. CoNLL 2003 dataset link contained):** https://towardsdatascience.com/named-entity-recognition-ner-with-bert-in-spark-nlp-874df20d1d77

In [2]:
def read_conll_2003_dataset(filepath):
    lines = []
    with open(filepath, 'r') as f:
        sentence = [] # initialize
        for i, line in enumerate(f.readlines()):
            if i < 2: # remove the first & second lines: i.e. '-DOCSTART- -X- -X- O\n' & '\n'
                continue

            data = line.split()

            if len(data) > 0:
                sentence.append((data[0], data[-1]))
            else:
                lines.append(sentence)
                sentence = [] # reset

    return lines

# if __name__=='__main__':    
#     for filepath in [
#         './conll_2003/eng.train',
#         './conll_2003/eng.testa',
#         './conll_2003/eng.testb',
#         ]:
#         lines = read_conll_2003_dataset(filepath)

#         print(f'{filepath} | len(lines) = {len(lines)} | {lines[:3]}')

In [3]:
import re

def convert_bert_sentences_labels(
    lines, encoder, decoder,
    idx_to_label_map = {0:'O', 1:'B-LOC', 2:'I-LOC', 3:'B-ORG', 4:'I-ORG', 5:'B-PER', 6:'I-PER', 7:'B-MISC', 8:'I-MISC'},
    verbose=False,
    ):
    label_to_idx_map = {v:k for k,v in idx_to_label_map.items()}
    
    #--------------------------------------------------------------------
    inputs, sentences, labels, outputs, decode_keys = [], [], [], [], []
    orig_sentences, orig_labels = [], []
    for line in lines:
        ins, words, tags, keys = [], [], [], [] # initialize
        orig_words, orig_tags = [], []
        for i, (word, tag) in enumerate(line):
            # nn = encoder(word)
            nn = encoder(' ' + word) # add a space because of the tokenization of gpt2
            tt = [decoder([n]) for n in nn]

            ins.extend(nn)
            words.extend(tt)
            if len(tt) == 1:
                tags.append(tag)
                keys.append(i)
            elif len(tt) > 1:
                tag_ = re.sub('^B-','I-',tag)
                tags.extend([tag] + [tag_]*(len(tt)-1))
                keys.extend([i] + [-1]*(len(tt)-1))
            else:
                raise(Exception(f'Brad error: an empty line is found'))
            orig_words.append(word)
            orig_tags.append(tag)

        inputs.append(ins)
        sentences.append(words)
        labels.append(tags)
        outputs.append([label_to_idx_map[t] for t in tags])
        decode_keys.append(keys)
        orig_sentences.append(orig_words)
        orig_labels.append(orig_tags)

    return {'inputs':inputs, 'sentences':sentences, 'labels':labels, 'outputs':outputs, 'decode_keys':decode_keys, 'orig_sentences':orig_sentences, 'orig_labels':orig_labels}

if __name__=='__main__':
    import tiktoken

    enc = tiktoken.get_encoding('gpt2')
    encoder = enc.encode
    decoder = enc.decode
    inputs = encoder('Hello world!!!')
    print(inputs)

    decoder([10185])

    for filepath in [
        './conll_2003/eng.train',
        # './conll_2003/eng.testa',
        # './conll_2003/eng.testb',
        ]:
        lines = read_conll_2003_dataset(filepath)

        print(f'{filepath} | len(lines) = {len(lines)} | {lines[:3]}')

        out = convert_bert_sentences_labels(lines, encoder, decoder)
        inputs, sentences, labels, outputs, decode_keys, orig_sentences, orig_labels = out['inputs'], out['sentences'], out['labels'], out['outputs'], out['decode_keys'], out['orig_sentences'], out['orig_labels']

        print(f'{filepath} | inputs = {inputs[:3]}')
        print(f'{filepath} | sentences = {sentences[:3]}')
        print(f'{filepath} | labels = {labels[:3]}')
        print(f'{filepath} | outputs = {outputs[:3]}')
        print(f'{filepath} | decode_keys = {decode_keys[:3]}')


In [10]:
# import re

# def convert_bert_sentences_labels(
#     lines, encoder, decoder,
#     idx_to_label_map = {0:'O', 1:'B-LOC', 2:'I-LOC', 3:'B-ORG', 4:'I-ORG', 5:'B-PER', 6:'I-PER', 7:'B-MISC', 8:'I-MISC'},
#     verbose=False,
#     ):
#     label_to_idx_map = {v:k for k,v in idx_to_label_map.items()}

#     inputs, outputs, sentences, labels = [], [], [], []
#     for line in lines:
#         #--- add
#         _words_, _ctags_, _ctags2_ = [], [], []
#         # for word, tag in zip(f,g):
#         for i, (word, tag) in enumerate(line):        
#             tag_ = re.sub('^B-','I-',tag)
#             if len(_words_) == 0: # if it's the first token
#             # if (len(_words_) == 0) or (word in [',', '.', "'s"]):
#                 _words_.append(word)
#                 _ctags_.extend([str(label_to_idx_map[tag])]*len(word))
#                 _ctags2_.extend([str(label_to_idx_map[tag])] + [str(label_to_idx_map[tag_])]*(len(word)-1)) # only initial character contains the true label (***)
#             else: # otherwise add a space before each token
#                 _words_.extend([' '] + [word])
#                 _ctags_.extend([' '] + [str(label_to_idx_map[tag])]*len(word))
#                 _ctags2_.extend([' '] + [str(label_to_idx_map[tag])] + [str(label_to_idx_map[tag_])]*(len(word)-1)) # only initial character contains the true label (***)
#         words = ''.join(_words_)
#         ctags = ''.join(_ctags_) # character tags
#         ctags2 = ''.join(_ctags2_) # character tags

#         if verbose:
#             print(words)
#             print(ctags)
#             print(ctags2)

#         #----------------------------------------------------------
#         encoded = encoder(words)
#         tokens = [decoder([t]) for t in encoded]    
#         tags_encoded, ttags, ix = [], [], 0
#         for t in tokens:
#             ttypes = set(ctags[ix:(ix+len(t))])
#             ttypes.discard(' ')

#             #---------------------------------------
#             if len(ttypes) == 1:
#                 pass
#             # elif (len(ttypes) == 2) and ('0' in ttypes):
#             #     pass
#             else:
#                 print(words)
#                 print(ctags)
#                 print(ctags2)
#                 print(tokens)
#                 raise(Exception('Brad error: more than one types...'))

#             #---------------------------------------
#             tag_num = int(ctags2[ix:(ix+len(t))].strip()[0]) # see only the initial character to pick the true label: cf. (***)
#             tags_encoded.append(tag_num)
#             ttags.append(idx_to_label_map[tag_num])
#             ix += len(t)

#         if verbose:
#             print(tokens)
#             print(ttags)

#         inputs.append(encoded)
#         outputs.append(tags_encoded)
#         sentences.append(tokens)
#         labels.append(ttags)
#     return {'inputs':inputs, 'outputs':outputs, 'sentences':sentences, 'labels':labels}

# # if __name__=='__main__':
# #     import tiktoken

# #     enc = tiktoken.get_encoding('gpt2')
# #     encoder = enc.encode
# #     decoder = enc.decode
# #     inputs = encoder('Hello world!!!')
# #     print(inputs)

# #     decoder([10185])

# #     for filepath in [
# #         './conll_2003/eng.train',
# #         # './conll_2003/eng.testa',
# #         # './conll_2003/eng.testb',
# #         ]:
# #         lines = read_conll_2003_dataset(filepath)

# #         print(f'{filepath} | len(lines) = {len(lines)} | {lines[:3]}')

# #         out = convert_bert_sentences_labels(lines, encoder, decoder)
# #         inputs, outputs, sentences, labels = out['inputs'], out['outputs'], out['sentences'], out['labels']

# #         print(f'{filepath} | inputs = {inputs[:3]}')
# #         print(f'{filepath} | outputs = {outputs[:3]}')
# #         print(f'{filepath} | sentences = {sentences[:3]}')
# #         print(f'{filepath} | labels = {labels[:3]}')


[15496, 995, 10185]
./conll_2003/eng.testb | len(lines) = 3683 | [[('SOCCER', 'O'), ('-', 'O'), ('JAPAN', 'B-LOC'), ('GET', 'O'), ('LUCKY', 'O'), ('WIN', 'O'), (',', 'O'), ('CHINA', 'B-PER'), ('IN', 'O'), ('SURPRISE', 'O'), ('DEFEAT', 'O'), ('.', 'O')], [('Nadim', 'B-PER'), ('Ladki', 'I-PER')], [('AL-AIN', 'B-LOC'), (',', 'O'), ('United', 'B-LOC'), ('Arab', 'I-LOC'), ('Emirates', 'I-LOC'), ('1996-12-06', 'O')]]
./conll_2003/eng.testb | inputs = [[15821, 4093, 1137, 532, 449, 2969, 1565, 17151, 406, 16696, 56, 25779, 11, 5870, 28893, 3268, 41016, 4805, 24352, 5550, 15112, 1404, 13], [45, 324, 320, 12862, 4106], [1847, 12, 29833, 11, 1578, 4498, 24880, 8235, 12, 1065, 12, 3312]]
./conll_2003/eng.testb | outputs = [[0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0], [5, 6, 6, 6, 6], [1, 2, 2, 0, 1, 2, 2, 0, 0, 0, 0, 0]]
./conll_2003/eng.testb | sentences = [['SO', 'CC', 'ER', ' -', ' J', 'AP', 'AN', ' GET', ' L', 'UCK', 'Y', ' WIN', ',', ' CH', 'INA', ' IN', ' SUR', 'PR

In [4]:
import torch
from torch.utils.data import DataLoader
import tiktoken
import os
import numpy as np

class CoNLL2003Dataset(torch.utils.data.Dataset): # https://pytorch.org/tutorials/beginner/data_loading_tutorial.html
    def __init__(self, split=['train','val','test'][0], block_size=20, data_root="./conll_2003"):
        self.block_size = block_size

        self.idx_to_label_map = {0:'O', 1:'B-LOC', 2:'I-LOC', 3:'B-ORG', 4:'I-ORG', 5:'B-PER', 6:'I-PER', 7:'B-MISC', 8:'I-MISC'}

        if split == 'train':
            filepath = os.path.join(data_root, 'eng.train')
        elif split == 'val':
            filepath = os.path.join(data_root, 'eng.testa')
        elif split == 'test':
            filepath = os.path.join(data_root, 'eng.testb')
        else:
            raise(Exception(f'Brad error: no such split options: split = {split} ...'))

        enc = tiktoken.get_encoding('gpt2')
        encoder = enc.encode
        decoder = enc.decode

        padding_input = enc.eot_token
        padding_output = 0
        self.padding_input = padding_input

        lines = read_conll_2003_dataset(filepath)
        out = convert_bert_sentences_labels(lines, encoder, decoder, idx_to_label_map=self.idx_to_label_map)
        inputs, outputs = out['inputs'], out['outputs']
        self.inputs = np.array([ii[:block_size] + [padding_input]*(block_size - len(ii)) for ii in inputs])
        self.outputs = np.array([ii[:block_size] + [padding_output]*(block_size - len(ii)) for ii in outputs])
        return

    def __len__(self):
        return len(self.inputs) # samples saved in hard drive

    def __getitem__(self, index):
        ins = self.inputs[index]
        outs = self.outputs[index]
        return {'in':ins, 'out':outs}

# if __name__=='__main__':
#     train_dataset = CoNLL2003Dataset(block_size=20, split='train')

#     ss = next(iter(train_dataset))
#     print(ss['in'])
#     print(ss['out'])

#     train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0, pin_memory=False)
#     # test_dataloader = DataLoader(testset, batch_size=1, shuffle=False, num_workers=opt.test_num_workers, pin_memory=True)

In [11]:
if __name__=='__main__':
    #--- dataset ----------------------------------------------------
    def prepare_dataloader(batch_size=1, block_size=20, data_root="./conll_2003", verbose=False):
        train_dataset = CoNLL2003Dataset(split='train', block_size=block_size, data_root=data_root)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        test_dataset = CoNLL2003Dataset(split='val', block_size=block_size, data_root=data_root)
        test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        return train_dataloader, test_dataloader

In [ ]:
if __name__=='__main__':
    train_dataloader, test_dataloader = prepare_dataloader(batch_size=2, block_size=20)
    print(len(train_dataloader))
    for ss in train_dataloader:
        break

In [ ]:
if __name__=='__main__':
    print(ss['in'].shape)
    print(ss['out'].shape)
    print(ss['in'])
    print(ss['out'])